In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau

from ugdatalab.models.galaxy_zoo import GalaxyZooDataset
from ugdatalab.models.galaxy_zoo.constants import N_LABELS
from ugdatalab.methods.cnn import train_cnn, count_parameters
from ugdatalab.methods.architectures import build_resnet18
from ugdatalab.methods.augmentation import RandomRotation360

import plotters

# Galaxy Image Classification — Optimization

This notebook improves the best model from NB 04 using two techniques:
1. **Task 20** — Learning rate scheduling (ReduceLROnPlateau)
2. **Task 21** — Data augmentation (random rotation by $\theta \in [0, 360)$ degrees)

Galaxy morphological labels are rotationally invariant — a spiral galaxy is still a spiral when rotated — so random rotation is a natural augmentation that generates valid training examples without changing labels. This effectively increases the size of the training set, reducing overfitting.

Target: RMSE $\leq 0.09$ (good), $\leq 0.08$ (excellent).

In [ ]:
# Load preprocessed data
img_data = np.load("galaxy_zoo_images.npz")
images = img_data["images"]
label_data = np.load("galaxy_zoo_labels.npz")
labels = label_data["labels"]
split_data = np.load("split_indices.npz")
train_idx, val_idx = split_data["train_idx"], split_data["val_idx"]

train_images, val_images = images[train_idx], images[val_idx]
train_labels, val_labels = labels[train_idx], labels[val_idx]
TARGET_SIZE = images.shape[1]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Determine best model from NB 04
custom_data = np.load("custom_result.npz", allow_pickle=True)
best_model_type = str(custom_data["best_model"])
print(f"Best model from NB 04: {best_model_type}")
print(f"Device: {DEVICE}")

## Task 20 — Learning Rate Scheduling

We use `ReduceLROnPlateau`: when the validation loss stops improving for `patience` epochs, the learning rate is reduced by a factor. This allows the optimizer to take large steps early in training (fast convergence) and small steps later (fine-tuning near the optimum).

We retrain the best model from scratch with the scheduler enabled. The two-panel plot shows loss curves (top) and the learning rate schedule (bottom), making it easy to identify whether loss reductions correspond to LR drops.

In [ ]:
train_ds = GalaxyZooDataset(train_images, train_labels, transform=None)
val_ds = GalaxyZooDataset(val_images, val_labels, transform=None)

model_sched = build_resnet18(n_labels=N_LABELS, input_size=TARGET_SIZE)

sched_result = train_cnn(
    model=model_sched,
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=64,
    n_epochs=50,
    lr=1e-3,
    device=DEVICE,
    seed=42,
    scheduler_factory=lambda opt: ReduceLROnPlateau(opt, factor=0.5, patience=3),
    num_workers=4,
)
print(f"Best epoch: {sched_result.best_epoch + 1}")
print(f"Best validation RMSE: {sched_result.best_val_loss:.4f}")

In [ ]:
axes = plotters.plot_loss_with_lr(
    sched_result.train_losses, sched_result.val_losses,
    sched_result.learning_rates, "ResNet-18 + LR Scheduler",
)
plt.show()

## Task 21 — Data Augmentation

We now add random rotation as a data augmentation transform. Each training image is rotated by a random angle $\theta \in [0, 360)$ degrees before cropping and being fed to the network. This exploits the rotational symmetry of galaxy morphological labels: a galaxy's classification does not depend on its orientation in the image.

The augmentation is applied *only* to the training set — the validation set remains unaugmented so that we measure generalization on fixed, unmodified images.

We retrain the best model with *both* augmentation and the learning rate scheduler.

In [ ]:
augment = RandomRotation360()
train_ds_aug = GalaxyZooDataset(train_images, train_labels, transform=augment)
val_ds_noaug = GalaxyZooDataset(val_images, val_labels, transform=None)

model_aug = build_resnet18(n_labels=N_LABELS, input_size=TARGET_SIZE)

aug_result = train_cnn(
    model=model_aug,
    train_dataset=train_ds_aug,
    val_dataset=val_ds_noaug,
    batch_size=64,
    n_epochs=50,
    lr=1e-3,
    device=DEVICE,
    seed=42,
    scheduler_factory=lambda opt: ReduceLROnPlateau(opt, factor=0.5, patience=3),
    num_workers=0,  # augmentation uses PIL, safer with 0 workers
)
print(f"Best epoch: {aug_result.best_epoch + 1}")
print(f"Best validation RMSE: {aug_result.best_val_loss:.4f}")

In [ ]:
axes = plotters.plot_loss_with_lr(
    aug_result.train_losses, aug_result.val_losses,
    aug_result.learning_rates, "ResNet-18 + Augmentation + LR Scheduler",
)
plt.show()

In [ ]:
# Save the best augmented model
torch.save(aug_result.model_state, "resnet18_augmented.pt")
np.savez_compressed(
    "augmented_result.npz",
    train_losses=aug_result.train_losses,
    val_losses=aug_result.val_losses,
    best_epoch=aug_result.best_epoch,
    best_val_loss=aug_result.best_val_loss,
    n_parameters=aug_result.n_parameters,
    learning_rates=aug_result.learning_rates,
)

np.savez_compressed(
    "scheduler_result.npz",
    train_losses=sched_result.train_losses,
    val_losses=sched_result.val_losses,
    best_epoch=sched_result.best_epoch,
    best_val_loss=sched_result.best_val_loss,
    n_parameters=sched_result.n_parameters,
    learning_rates=sched_result.learning_rates,
)
print("Saved resnet18_augmented.pt, augmented_result.npz, scheduler_result.npz")